# 6교시. OCR 및 정보 추출 기능 연동

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/leecks1119/document_ai_lecture/blob/document_ai_lecture_2026/colab/06_ocr_ai_integration.ipynb)

**이번 교시 행동:** 업로드한 파일을 실제 OCR 함수에 연결하고 LIVE·오류·복구 모드를 화면에서 구분합니다.

**통과 증거:** `course_outputs/app_06.py`

> Google Colab도 외부 클라우드입니다. 조직 승인 없는 개인·회사 문서는
> 업로드하지 않습니다. 필수 실습은 저장소의 비식별 공개·합성 샘플만
> 사용합니다.

화면의 실행 모드를 먼저 확인합니다.

- `LIVE`: 현재 파일에 실제 모델을 실행한 결과
- `PREPARED_FALLBACK`: 공개 샘플을 사람이 검수해 둔 복구 결과
- 3분 이상 멈추면 실행을 중지하고 복구 결과로 계속합니다.
- 각 교시 끝에서 `CHECKPOINT PASS`와 산출물 파일을 확인합니다.


In [ ]:
import json
import os
import platform
import sys
from pathlib import Path

OUTPUT_DIR = Path("course_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)
VALIDATION_MODE = os.getenv("COURSE_VALIDATE_PREPARED") == "1"

def upload_previous_artifact(filename):
    target = OUTPUT_DIR / filename
    if target.exists() or VALIDATION_MODE:
        return target if target.exists() else None
    try:
        from google.colab import files
    except ImportError:
        return None
    print(f"이전 교시에서 내려받은 {filename}을 선택하세요.")
    uploaded = files.upload()
    if filename not in uploaded:
        raise FileNotFoundError(
            f"{filename}이 선택되지 않았습니다. 준비 입력을 쓰려면 "
            "USE_PREPARED_INPUT=True로 바꾸세요."
        )
    target.write_bytes(uploaded[filename])
    return target


def download_artifact(path):
    if VALIDATION_MODE:
        return
    try:
        from google.colab import files
    except ImportError:
        return
    files.download(str(path))

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("공통 작업 폴더:", OUTPUT_DIR.resolve())


In [ ]:
import importlib.metadata
import subprocess

required_streamlit = "1.60.0"
try:
    installed_streamlit = importlib.metadata.version("streamlit")
except importlib.metadata.PackageNotFoundError:
    installed_streamlit = None
if installed_streamlit != required_streamlit:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", f"streamlit=={required_streamlit}"]
    )


In [ ]:
from textwrap import dedent

app_code = 'import tempfile\nfrom pathlib import Path\nimport streamlit as st\n\nGOLDEN_OCR_TEXT = \'이태리집\\n거래일시 2025-10-04 12:33:37\\n페퍼로니 앤 치즈 29,000 1 29,000\\n토마토 파스타 14,000 1 14,000\\n수제 돈가스 13,000 1 13,000\\n새우 칠리치 필라 14,000 1 14,000\\n콜라 2,000 3 6,000\\n합계 금액 76,000\\n부가세 과세물품가액 69,094\\n부가세 6,906\\n\'\nGOLDEN_RECEIPT = {\'document_type\': \'receipt\', \'store_name\': \'이태리집\', \'date\': \'2025-10-04\', \'total_amount\': 76000, \'items\': [{\'name\': \'페퍼로니 앤 치즈\', \'quantity\': 1, \'unit_price\': 29000, \'line_total\': 29000}, {\'name\': \'토마토 파스타\', \'quantity\': 1, \'unit_price\': 14000, \'line_total\': 14000}, {\'name\': \'수제 돈가스\', \'quantity\': 1, \'unit_price\': 13000, \'line_total\': 13000}, {\'name\': \'새우 칠리치 필라\', \'quantity\': 1, \'unit_price\': 14000, \'line_total\': 14000}, {\'name\': \'콜라\', \'quantity\': 3, \'unit_price\': 2000, \'line_total\': 6000}], \'adjustments\': {\'discount\': 0, \'tax\': 0, \'service\': 0, \'rounding\': 0}, \'tax_breakdown\': {\'mode\': \'included_in_item_prices\', \'supply_amount\': 69094, \'vat\': 6906, \'payable_total\': 76000}, \'raw_values\': {\'store_name\': \'이태리집\', \'date\': \'2025-10-04 12:33:37\', \'total_amount\': \'76,000\'}, \'cleaned_values\': {\'store_name\': \'이태리집\', \'date\': \'2025-10-04\', \'total_amount\': 76000}, \'evidence\': {\'store_name\': {\'raw_value\': \'이태리집\', \'line\': 1}, \'date\': {\'raw_value\': \'거래일시 2025-10-04 12:33:37\', \'line\': 2}, \'total_amount\': {\'raw_value\': \'합계 금액 76,000\', \'line\': 8}}, \'source_mode\': \'prepared_fixture_rule_extraction\'}\n\nimport re\n\ndef to_int(value):\n    return int(value.replace(",", ""))\n\n\ndef extract_receipt_from_text(text, source_mode):\n    lines = [line.strip() for line in text.splitlines() if line.strip()]\n    date_match = re.search(r"\\b(\\d{4})[-./](\\d{1,2})[-./](\\d{1,2})\\b", text)\n    total_line = next(\n        (\n            line\n            for line in lines\n            if re.search(r"(?:합\\s*계|결제\\s*금액|총\\s*액)", line)\n        ),\n        None,\n    )\n    total_candidates = (\n        re.findall(r"(?<![\\d,])\\d[\\d,]*(?![\\d,])", total_line)\n        if total_line\n        else []\n    )\n    total_raw = total_candidates[-1] if total_candidates else None\n    supply_match = re.search(\n        r"(?:부가세\\s*)?과세물품가액\\s*[:：]?\\s*([\\d,]+)",\n        text,\n    )\n    vat_match = re.search(\n        r"^부가세(?!\\s*과세물품가액)\\s*[:：]?\\s*([\\d,]+)",\n        text,\n        re.MULTILINE,\n    )\n    item_pattern = re.compile(\n        r"^(?P<name>.+?)\\s+(?P<unit>[\\d,]+)\\s+"\n        r"(?P<quantity>\\d+)\\s+(?P<line>[\\d,]+)$"\n    )\n    items = []\n    item_evidence = []\n    for line_number, line in enumerate(lines, start=1):\n        match = item_pattern.search(line)\n        if match:\n            item = {\n                "name": match.group("name"),\n                "quantity": int(match.group("quantity")),\n                "unit_price": to_int(match.group("unit")),\n                "line_total": to_int(match.group("line")),\n            }\n            items.append(item)\n            item_evidence.append({"line": line_number, "raw_value": line})\n\n    date_value = (\n        f"{int(date_match.group(1)):04d}-{int(date_match.group(2)):02d}-"\n        f"{int(date_match.group(3)):02d}"\n        if date_match else None\n    )\n    total_value = to_int(total_raw) if total_raw else None\n    supply_value = to_int(supply_match.group(1)) if supply_match else None\n    vat_value = to_int(vat_match.group(1)) if vat_match else None\n    return {\n        "document_type": "receipt",\n        "store_name": lines[0] if lines else None,\n        "date": date_value,\n        "total_amount": total_value,\n        "items": items,\n        "adjustments": {"discount": 0, "tax": 0, "service": 0, "rounding": 0},\n        "tax_breakdown": {\n            "mode": "included_in_item_prices",\n            "supply_amount": supply_value,\n            "vat": vat_value,\n            "payable_total": total_value,\n        } if supply_value is not None and vat_value is not None else None,\n        "raw_values": {\n            "store_name": lines[0] if lines else None,\n            "date": date_match.group(0) if date_match else None,\n            "total_amount": total_raw,\n        },\n        "cleaned_values": {\n            "store_name": lines[0] if lines else None,\n            "date": date_value,\n            "total_amount": total_value,\n        },\n        "evidence": {\n            "store_name": {"line": 1, "raw_value": lines[0] if lines else None},\n            "date": {"raw_value": date_match.group(0) if date_match else None},\n            "total_amount": {"raw_value": total_line},\n            "items": item_evidence,\n        },\n        "source_mode": source_mode,\n    }\n\ndef run_live_ocr(uploaded):\n    suffix = Path(uploaded.name).suffix.lower()\n    with tempfile.NamedTemporaryFile(suffix=suffix, delete=False) as temp:\n        temp.write(uploaded.getvalue())\n        path = temp.name\n    try:\n        from paddleocr import PaddleOCR\n        engine = PaddleOCR(\n            lang="korean",\n            ocr_version="PP-OCRv5",\n            use_doc_orientation_classify=False,\n            use_doc_unwarping=False,\n            use_textline_orientation=False,\n            device="cpu",\n        )\n        page = list(engine.predict(path))[0]\n        payload = page.json() if callable(page.json) else page.json\n        result = payload.get("res", payload)\n        return "\\n".join(result.get("rec_texts", []))\n    finally:\n        Path(path).unlink(missing_ok=True)\n\n\ndef process_document(uploaded=None, *, use_prepared=False):\n    if use_prepared:\n        text = GOLDEN_OCR_TEXT\n        mode = "PREPARED_FALLBACK"\n    elif uploaded is None:\n        return {"ok": False, "mode": "INPUT_ERROR", "error": "파일을 선택하세요."}\n    else:\n        try:\n            text = run_live_ocr(uploaded)\n            mode = "LIVE"\n        except Exception as exc:\n            return {\n                "ok": False,\n                "mode": "LIVE_ERROR",\n                "error": f"{type(exc).__name__}: {exc}",\n                "recovery": "공개 샘플 준비 결과 버튼을 선택하세요.",\n            }\n    data = extract_receipt_from_text(\n        text,\n        "live_ocr_rule_extraction" if mode == "LIVE"\n        else "prepared_fixture_rule_extraction",\n    )\n    return {"ok": True, "mode": mode, "ocr_text": text, "data": data}\n\n\nst.title("영수증 Document AI 연결 앱")\nuploaded = st.file_uploader(\n    "승인된 비식별 이미지 또는 PDF 한 장 · 최대 5MB",\n    type=["png", "jpg", "jpeg", "pdf"],\n    max_upload_size=5,\n    help="PNG, JPG, JPEG, PDF만 허용합니다. 수업에서는 한 번에 5MB 이하 한 장만 처리합니다.",\n)\nleft, right = st.columns(2)\nrun_live = left.button("업로드 파일 LIVE 처리")\nrun_prepared = right.button("공개 샘플 준비 결과")\nif run_live or run_prepared:\n    result = process_document(uploaded, use_prepared=run_prepared)\n    if result["ok"]:\n        st.success(f"실행 모드: {result[\'mode\']}")\n        st.text_area("OCR 원문", result["ocr_text"], height=220)\n        st.json(result["data"])\n    else:\n        st.error(f"{result[\'mode\']} · {result[\'error\']}")\n        if result.get("recovery"):\n            st.info(result["recovery"])\n'
output_path = OUTPUT_DIR / "app_06.py"
output_path.write_text(app_code, encoding="utf-8")
print("저장:", output_path)


In [ ]:
from streamlit.testing.v1 import AppTest

app_test = AppTest.from_file(str(output_path)).run(timeout=20)
assert not app_test.exception
assert app_test.title[0].value == "영수증 Document AI 연결 앱"
assert len(app_test.button) == 2
app_test.button[1].click().run(timeout=20)
assert any("PREPARED_FALLBACK" in item.value for item in app_test.success)
assert app_test.json
print("CHECKPOINT 1/1 PASS: 앱 연결·모드 표시·JSON 출력")
